# Day 01：卷积的诞生 —— 让机器学会看图

> 👁️ 第三周 · 视觉的征服 · 第 1 天

上周我们用 MLP 成功解决了 XOR 问题，证明了多层网络的能力。但当我们把 MLP 用在图像上时，灾难发生了——参数多到爆炸，而且图片里的猫稍微挪个位置，模型就认不出来了。

今天，我们要学习一个改变计算机视觉历史的概念：**卷积（Convolution）**。它用两个简单的想法——「只看局部」和「共享参数」——同时解决了参数爆炸和平移识别两大难题。

**今天的任务**：理解卷积运算的本质，手写一个 2D 卷积，亲眼看到它如何检测图像中的边缘。

---
## 1. 历史背景：当神经网络遇上图像

### MLP 看图的灾难

假设我们有一张 28×28 的灰度手写数字图片（就是 MNIST 数据集里的那种）。如果用 MLP 来处理它：

**第一步：把图片「拍扁」**

MLP 只能接收一维向量，所以我们必须把 28×28 = 784 个像素排成一行。这就像把一张照片撕成 784 个小碎片，然后排成一排——**空间结构完全丢失了**。

原本在图片中相邻的像素（比如数字「8」的左上角曲线），在拍扁后可能被分隔得很远。MLP 根本不知道哪些像素原本是邻居。

**第二步：参数爆炸**

假设隐藏层有 128 个神经元：
- 输入层 → 隐藏层：784 × 128 = **100,352** 个参数
- 隐藏层 → 输出层：128 × 10 = 1,280 个参数
- 总计：**超过 10 万个参数**，仅仅是一张 28×28 的小图片！

如果是 224×224 的彩色图片（3 通道），输入就有 224×224×3 = 150,528 个像素。连接到 128 个隐藏神经元就需要 **1900 多万个参数**。这还没算后续的层。

**第三步：平移失忆症**

更致命的问题是：MLP 对位置极其敏感。如果训练时猫都在图片正中央，测试时猫跑到了右下角——MLP 就完全不认识了。

因为在 MLP 眼里，「左上角像素」和「右下角像素」是完全不同的输入，它学到的「猫在中央」的模式，对「猫在角落」毫无帮助。

### 生物学灵感：Hubel 和 Wiesel 的发现

1959 年（比感知机还早！），两位神经科学家 David Hubel 和 Torsten Wiesel 做了一个改变历史的实验。

他们把微电极插入猫的视觉皮层，然后给猫看各种图案。他们发现：

- **简单细胞（Simple Cells）**：只对特定方向的边缘有反应。有的细胞只对「竖线」兴奋，有的只对「横线」兴奋，有的只对特定角度的斜线兴奋。
- **复杂细胞（Complex Cells）**：对边缘的方向敏感，但对边缘的**具体位置**不那么敏感——边缘稍微移动一点，它照样有反应。
- **感受野（Receptive Field）**：每个视觉皮层细胞只「看」视野中的一小块区域，而不是整个视野。

这三个发现——**局部感知、方向选择性、平移容忍**——后来全部被卷积神经网络继承了。

Hubel 和 Wiesel 因此获得了 1981 年的诺贝尔生理学或医学奖。

### 从生物学到算法：福岛邦彦的 Neocognitron

1980 年，日本科学家福岛邦彦（Kunihiko Fukushima）受到 Hubel 和 Wiesel 的启发，提出了 **Neocognitron**——这是世界上第一个卷积神经网络的雏形。

Neocognitron 的核心思想：
- **S 细胞（Simple Cells）**：类似今天的卷积层，提取局部特征
- **C 细胞（Complex Cells）**：类似今天的池化层，提供平移不变性
- **层级结构**：S 层 → C 层 → S 层 → C 层……逐层提取越来越抽象的特征

但 Neocognitron 有一个致命缺陷：**它没有反向传播**。福岛邦彦用的是无监督的「自学」规则，效果有限。

真正的突破要等到 1989 年——Yann LeCun 把反向传播和卷积结合起来，创造了我们今天熟知的 CNN。

### 卷积的三个核心思想

在深入数学之前，先理解卷积为什么能解决 MLP 的三大困境：

| MLP 的困境 | 卷积的解法 | 核心思想 |
|-----------|-----------|---------|
| 参数爆炸（784×128=10万） | 卷积核只有 3×3=9 个参数 | **权值共享**：整张图用同一个卷积核 |
| 空间结构丢失（拍扁） | 保留 2D 结构，在图上滑动 | **局部感受野**：每次只看一小块 |
| 平移失忆症 | 同一个卷积核扫过整张图 | **平移等变性**：猫在哪都能检测到 |

这三个思想——**局部感受野、权值共享、平移等变性**——是卷积的三大支柱。接下来我们逐一展开。

---
## 2. 生活隐喻：用放大镜检查名画

### 隐喻一：放大镜扫描

想象你拿一个放大镜检查一幅名画，判断它是不是真迹。

**MLP 的做法**（全局扫描）：
- 你站在画前面，一眼看完整幅画
- 然后凭「整体感觉」判断真伪
- 问题：如果画稍微歪了一点，你的「整体感觉」就全变了

**卷积的做法**（局部扫描）：
- 你拿一个放大镜，从画的左上角开始
- 每次只看放大镜覆盖的那一小块区域（比如 3cm×3cm）
- 检查完这一块后，把放大镜往右挪一点，再看下一块
- 从左到右、从上到下，逐块扫描完整幅画
- 最后综合所有局部发现，做出判断

关键洞察：
- **放大镜的大小**就是「卷积核大小」（kernel size），比如 3×3
- **每次挪动的距离**就是「步长」（stride），比如 1 个像素
- **放大镜本身**就是「卷积核」（kernel），它定义了你在找什么特征
- **扫描完整幅画**就是「卷积运算」

同一个放大镜扫遍整幅画——这就是**权值共享**。你不需要给画的每个位置配一个不同的放大镜。

### 隐喻二：拼图匹配

另一个理解卷积的好方式是「拼图匹配」：

想象你有一张完整的拼图（输入图像），和一小块「模板」（卷积核）。你的任务是：**在拼图的每个位置，检查这一小块和模板有多相似**。

- 把模板放在拼图的左上角，逐格对比，算一个「相似度分数」
- 把模板往右挪一格，再算一次
- 重复，直到覆盖整张拼图
- 最终得到一张「相似度热力图」——哪里和模板最像，哪里的分数就最高

如果模板是一个「竖边检测器」（左边暗、右边亮），那么：
- 在图像的平坦区域（全是灰色），相似度 ≈ 0
- 在竖边位置（左边暗右边亮），相似度很高
- 在横边位置（上边暗下边亮），相似度 ≈ 0

这就是卷积的本质：**用一个模板在图像上滑动，在每个位置计算模板与图像局部区域的匹配程度**。

### 隐喻三：CT 扫描

如果你做过 CT 检查，卷积的原理就更好理解了。

CT 扫描不是一次性拍完整个人体，而是：
1. 用一束很窄的 X 光从不同角度扫描
2. 每次只扫描一个「切片」
3. 把所有切片的信息综合起来，重建出 3D 图像

卷积核就像那束 X 光——它很窄（只看局部），但通过滑动覆盖了整个图像。每个位置得到一个「响应值」，所有响应值组成一张新的「特征图」。

---
## 3. 数学直觉：卷积到底在算什么？

### 一维卷积：从最简单的情况开始

在进入二维图像卷积之前，先用一维信号理解卷积的本质。

假设你有一个一维信号（比如一段音频的波形）：
```
信号: [2, 4, 1, 7, 3, 0, 5]
```

和一个卷积核（你想检测的模式）：
```
卷积核: [1, 0, -1]  （这是一个「变化检测器」）
```

卷积的计算过程——把卷积核在信号上滑动，每个位置做「逐元素相乘再求和」：

```
位置 0: 核[1,0,-1] 对齐 信号[2,4,1]
        = 1×2 + 0×4 + (-1)×1 = 2 + 0 - 1 = 1

位置 1: 核[1,0,-1] 对齐 信号[4,1,7]
        = 1×4 + 0×1 + (-1)×7 = 4 + 0 - 7 = -3

位置 2: 核[1,0,-1] 对齐 信号[1,7,3]
        = 1×1 + 0×7 + (-1)×3 = 1 + 0 - 3 = -2

位置 3: 核[1,0,-1] 对齐 信号[7,3,0]
        = 1×7 + 0×3 + (-1)×0 = 7 + 0 - 0 = 7

位置 4: 核[1,0,-1] 对齐 信号[3,0,5]
        = 1×3 + 0×0 + (-1)×5 = 3 + 0 - 5 = -2
```

输出: `[1, -3, -2, 7, -2]`

观察规律：核 `[1, 0, -1]` 计算的是「右边减左边」。正值表示信号在上升，负值表示信号在下降。这就是一个**边缘检测器**！

### 二维卷积：图像的滑动窗口

把一维卷积扩展到二维，就是图像卷积。原理完全一样——**滑动窗口 + 逐元素相乘 + 求和**。

假设有一张 5×5 的灰度图像：
```
    0  1  2  3  4   ← 列索引
0  [0, 0, 0, 0, 0]
1  [0, 1, 1, 1, 0]   ← 中间有一个白色方块
2  [0, 1, 1, 1, 0]
3  [0, 1, 1, 1, 0]
4  [0, 0, 0, 0, 0]
```

和一个 3×3 的卷积核（竖边检测器）：
```
[-1,  0,  1]
[-1,  0,  1]
[-1,  0,  1]
```

这个核的含义是：「左边暗（×-1）、中间无所谓（×0）、右边亮（×1）」。如果图像中某个位置左边暗右边亮，这个核的输出就会很大。

把核放在图像的 (1,1) 位置（即白色方块的左上角）：

```
图像局部 (1,1) 处:     卷积核:
[0, 0, 0]              [-1,  0,  1]
[0, 1, 1]              [-1,  0,  1]
[0, 1, 1]              [-1,  0,  1]

逐元素相乘再求和:
= (-1)×0 + 0×0 + 1×0
+ (-1)×0 + 0×1 + 1×1
+ (-1)×0 + 0×1 + 1×1
= 0 + 0 + 0 + 0 + 0 + 1 + 0 + 0 + 1
= 2
```

输出 2 表示这里检测到了「左边暗右边亮」的模式——正是白色方块的左边缘！

### 卷积的数学定义

用数学公式表达二维卷积：

$$(I * K)(i, j) = \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} I(i+m, j+n) \cdot K(m, n)$$

其中：
- $I$ 是输入图像（Image）
- $K$ 是卷积核（Kernel），大小为 $k_h \times k_w$
- $(i, j)$ 是输出位置
- $*$ 是卷积运算符

用大白话说：**对于输出位置 (i, j)，取图像中以 (i, j) 为左上角、和卷积核同样大小的一块区域，与卷积核逐元素相乘后全部加起来**。

### 输出尺寸的计算

卷积后输出会变小。如果输入尺寸为 $H \times W$，卷积核尺寸为 $k_h \times k_w$，不使用填充（padding）：

$$H_{out} = H - k_h + 1$$
$$W_{out} = W - k_w + 1$$

例如：5×5 的图像，用 3×3 的卷积核 → 输出是 3×3。

如果使用填充（padding），在图像四周补零：

$$H_{out} = H + 2p - k_h + 1$$

例如：5×5 的图像，padding=1（四周各补一圈 0），3×3 的卷积核 → 输出是 5×5（和输入一样大）。

### 经典卷积核：Sobel 算子

在深度学习之前，计算机视觉研究者手工设计了很多卷积核来检测特定特征。最著名的是 **Sobel 算子**：

**Sobel-X（检测竖边）**：
```
[-1,  0,  1]
[-2,  0,  2]
[-1,  0,  1]
```

**Sobel-Y（检测横边）**：
```
[-1, -2, -1]
[ 0,  0,  0]
[ 1,  2,  1]
```

Sobel-X 检测「左右差异」——左边暗右边亮时输出正值。Sobel-Y 检测「上下差异」——上边暗下边亮时输出正值。

把 Sobel-X 和 Sobel-Y 的结果结合起来，就能得到边缘的**强度**和**方向**：

$$G = \sqrt{G_x^2 + G_y^2}$$

$$\theta = \arctan\left(\frac{G_y}{G_x}\right)$$

下面用代码验证这些手工设计的卷积核，亲眼看看它们如何检测边缘。

---
## 4. 代码实现：手写 2D 卷积

现在用代码实现二维卷积。核心逻辑就是三层循环：
- 外层循环：在图像高度上滑动
- 中层循环：在图像宽度上滑动
- 内层循环：逐元素相乘再求和

每个代码 cell 都是**独立的**——包含完整的 import 和数据定义，可以单独运行。

In [ ]:
# 设置中文字体
import matplotlib.pyplot as plt
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

import torch
import matplotlib.pyplot as plt
import numpy as np

# 模拟一张简单的「图片」：黑白渐变
# 想象这是照片的一小部分
image = torch.tensor([
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],  # 白色方块（从上到下渐变）
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0]
], dtype=torch.float32)

print("一张简单的「图片」：")
print(image)
print(f"\n图片尺寸: {image.shape}")
print("\n想象这是画面中的一道边缘——左边是背景（暗），右边是物体（亮）")

### 手写卷积函数

下面实现一个最朴素的 2D 卷积——不做任何优化，只求逻辑清晰。

In [ ]:
import torch

def conv2d_naive(image, kernel):
    """最朴素的 2D 卷积实现：滑动窗口 + 逐元素相乘 + 求和。

    参数:
        image: 输入图像，形状 [H, W]
        kernel: 卷积核，形状 [kH, kW]
    返回:
        output: 卷积输出，形状 [H-kH+1, W-kW+1]
    """
    H, W = image.shape  # 图像的高度和宽度
    kH, kW = kernel.shape  # 卷积核的高度和宽度

    # 计算输出尺寸（无 padding 时输出会变小）
    out_H = H - kH + 1  # 输出高度
    out_W = W - kW + 1  # 输出宽度

    # 创建输出张量，初始化为 0
    output = torch.zeros(out_H, out_W)

    # 在图像上滑动卷积核
    for i in range(out_H):  # 遍历输出的每一行
        for j in range(out_W):  # 遍历输出的每一列
            # 取出图像中以 (i,j) 为左上角、和卷积核同样大小的区域
            region = image[i:i+kH, j:j+kW]
            # 逐元素相乘再求和：region 和 kernel 对应位置相乘，全部加起来
            output[i, j] = torch.sum(region * kernel)

    return output

# 定义 Sobel-X 卷积核（检测竖边）
sobel_x = torch.tensor([
    [-1.0, 0.0, 1.0],
    [-2.0, 0.0, 2.0],
    [-1.0, 0.0, 1.0]
])

# 定义 Sobel-Y 卷积核（检测横边）
sobel_y = torch.tensor([
    [-1.0, -2.0, -1.0],
    [ 0.0,  0.0,  0.0],
    [ 1.0,  2.0,  1.0]
])

# 对图像应用卷积
image = torch.tensor([
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0]
])

edge_x = conv2d_naive(image, sobel_x)  # 检测竖边
edge_y = conv2d_naive(image, sobel_y)  # 检测横边

print("Sobel-X 卷积结果（检测竖边）：")
print(edge_x)
print("\n解读：正值 = 左边暗右边亮（白色方块的左边缘）")
print("      负值 = 左边亮右边暗（白色方块的右边缘）")

print("\nSobel-Y 卷积结果（检测横边）：")
print(edge_y)
print("\n解读：正值 = 上边暗下边亮（白色方块的上边缘）")
print("      负值 = 上边亮下边暗（白色方块的下边缘）")

### 可视化卷积效果

用图片直观展示原始图像和卷积后的边缘检测结果。

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

def conv2d_naive(image, kernel):
    """最朴素的 2D 卷积实现"""
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            region = image[i:i+kH, j:j+kW]
            output[i, j] = torch.sum(region * kernel)
    return output

# 创建一张更丰富的测试图像——模拟一个「T」字形
image = torch.zeros(7, 7)
image[1, 2:5] = 1.0  # T 的横杠
image[2:6, 3] = 1.0  # T 的竖杠

# Sobel 卷积核
sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]])

# 卷积
edge_x = conv2d_naive(image, sobel_x)
edge_y = conv2d_naive(image, sobel_y)
edge_magnitude = torch.sqrt(edge_x**2 + edge_y**2)  # 边缘强度

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image.numpy(), cmap='gray')
axes[0].set_title('原始图像（T 字形）')
axes[0].axis('off')

axes[1].imshow(edge_x.numpy(), cmap='RdBu', vmin=-4, vmax=4)
axes[1].set_title('Sobel-X（竖边检测）')
axes[1].axis('off')

axes[2].imshow(edge_y.numpy(), cmap='RdBu', vmin=-4, vmax=4)
axes[2].set_title('Sobel-Y（横边检测）')
axes[2].axis('off')

axes[3].imshow(edge_magnitude.numpy(), cmap='hot')
axes[3].set_title('边缘强度 √(Gx²+Gy²)')
axes[3].axis('off')

plt.suptitle('卷积的边缘检测效果', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day01_edge_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print("观察要点：")
print("  Sobel-X 检测到了 T 字的左右边缘（竖边）")
print("  Sobel-Y 检测到了 T 字的上下边缘（横边）")
print("  边缘强度综合了两个方向，完整勾勒出了 T 字的轮廓")

### 多个卷积核同时检测

真实场景中，我们不会只用一个卷积核。不同的卷积核检测不同的特征——有的检测竖边，有的检测横边，有的检测斜边，有的检测角点……

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

def conv2d_naive(image, kernel):
    """最朴素的 2D 卷积实现"""
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            region = image[i:i+kH, j:j+kW]
            output[i, j] = torch.sum(region * kernel)
    return output

# 创建测试图像——一个「十」字形
image = torch.zeros(9, 9)
image[3, 1:8] = 1.0  # 横杠
image[1:8, 4] = 1.0  # 竖杠

# 定义多个不同的卷积核
kernels = {
    '竖边检测': torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]),
    '横边检测': torch.tensor([[-1.,-2.,-1.], [ 0., 0., 0.], [ 1., 2., 1.]]),
    '斜边检测(↘)': torch.tensor([[ 2., 1., 0.], [ 1., 0.,-1.], [ 0.,-1.,-2.]]),
    '斜边检测(↗)': torch.tensor([[ 0., 1., 2.], [-1., 0., 1.], [-2.,-1., 0.]]),
    '模糊（均值）': torch.ones(3, 3) / 9.0,  # 3x3 均值滤波
    '锐化': torch.tensor([[ 0.,-1., 0.], [-1., 5.,-1.], [ 0.,-1., 0.]]),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# 显示原图
axes[0].imshow(image.numpy(), cmap='gray')
axes[0].set_title('原始图像（十字形）')
axes[0].axis('off')

# 显示每个卷积核的效果
for idx, (name, kernel) in enumerate(kernels.items()):
    result = conv2d_naive(image, kernel)
    ax = axes[idx + 1]
    im = ax.imshow(result.numpy(), cmap='RdBu', vmin=-4, vmax=4)
    ax.set_title(name)
    ax.axis('off')

axes[7].axis('off')  # 最后一个位置留空
plt.suptitle('不同卷积核检测不同特征', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day01_multiple_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

print("关键洞察：")
print("  不同的卷积核 = 不同的「视角」")
print("  同一个图像，用不同核看，看到的东西完全不同")
print("  这就是 CNN 第一层做的事情——用多个卷积核从不同角度观察图像")

---
## 5. 验证实验：卷积 vs 全连接

用一个具体的对比实验来验证卷积的优势。我们构造一个简单的任务：**检测图像中是否有竖边**。

对比两种方案：
- **全连接方案**：把 5×5 图像拍扁成 25 维向量，接一个输出神经元
- **卷积方案**：用一个 3×3 的 Sobel-X 卷积核扫描图像，取最大值作为输出

In [ ]:
import torch

# 测试图像：有的有竖边，有的没有
test_images = {
    '纯色背景（无边）': torch.zeros(5, 5),
    '竖边在中间': torch.tensor([
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
    ]),
    '竖边在左边': torch.tensor([
        [1., 1., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
    ]),
    '横边（无竖边）': torch.tensor([
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
    ]),
}

def conv2d_naive(image, kernel):
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            region = image[i:i+kH, j:j+kW]
            output[i, j] = torch.sum(region * kernel)
    return output

sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])

print("卷积方案：用 Sobel-X 检测竖边")
print("=" * 55)
print(f"{'图像描述':<16} {'最大响应值':<12} {'有竖边？'}")
print("-" * 55)

for name, img in test_images.items():
    edge_map = conv2d_naive(img, sobel_x)
    max_response = edge_map.abs().max().item()
    has_vertical = "是" if max_response > 1.0 else "否"
    print(f"{name:<16} {max_response:<12.1f} {has_vertical}")

print("\n关键洞察：")
print("  卷积方案只需要 9 个参数（3×3 卷积核），就能检测竖边")
print("  全连接方案需要 25+1=26 个参数，而且对位置敏感")
print("  如果把竖边从中间移到左边，全连接需要重新学习，卷积不需要！")

---
## 翻译词典

| 生活直觉 | 深度学习术语 |
|----------|-------------|
| 放大镜的大小 | 卷积核大小（Kernel Size） |
| 每次挪动的距离 | 步长（Stride） |
| 在图像四周补零 | 填充（Padding） |
| 放大镜本身（检测模板） | 卷积核（Kernel / Filter） |
| 扫描完整幅画 | 卷积运算（Convolution） |
| 扫描后得到的「相似度图」 | 特征图（Feature Map） |
| 同一个放大镜扫遍全图 | 权值共享（Weight Sharing） |
| 每次只看一小块 | 局部感受野（Local Receptive Field） |
| 猫挪了位置照样认识 | 平移等变性（Translation Equivariance） |
| Sobel 核检测竖边 | 手工设计特征（Hand-crafted Feature） |

---
## 今日总结

| 概念 | 直觉理解 |
|------|----------|
| 卷积 | 用一个小模板在图像上滑动，每个位置算一个匹配分数 |
| 卷积核 | 定义你在找什么特征的「模板」 |
| 权值共享 | 整张图用同一个卷积核，参数数量与图像大小无关 |
| 局部感受野 | 每个输出像素只依赖输入的一小块区域 |
| 平移等变性 | 特征出现在图像的任何位置，卷积核都能检测到 |
| Sobel 算子 | 经典的手工设计卷积核，用于边缘检测 |

**卷积的三大支柱**：
- **局部感受野**：不看全局，只看局部 → 保留空间结构
- **权值共享**：同一个核扫遍全图 → 参数极少
- **平移等变性**：特征在哪都能检测 → 位置不敏感

**明日预告**：今天我们用的是手工设计的卷积核（Sobel）。但现实中的图像千变万化，手工设计所有核是不可能的。明天学习**卷积层（nn.Conv2d）**——让卷积核自己从数据中学习该检测什么特征！